# A/B-Test-Analyse – KI im Controlling

**Ziel:** Auswertung eines A/B-Test-Datensatzes aus drei Controlling-Szenarien  
**Datenquelle:** `Beispieldaten_AB_Test.csv` (im gleichen Ordner)  
**Szenarien:** Marketing, Pricing, Prozessoptimierung  
**Gruppen:** A (Kontrollgruppe) vs. B (Testgruppe)

---

**Wie dieser Code entstanden ist:**  
Dieser Code wurde von Claude (claude.ai) generiert auf Basis des Prompts:  
> *"Erstelle einen Python-Code für einen A/B-Test-Vergleich mit t-Test und Plotly-Visualisierung..."*

**Abhängigkeiten:** pandas, scipy, plotly, IPython

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import warnings
warnings.filterwarnings('ignore')

# Third Party
import pandas as pd                    # Data manipulation
import numpy as np                     # Numerical operations
from scipy import stats                # Statistical tests
import plotly.graph_objects as go      # Interactive charts
import plotly.express as px            # Quick charts
from plotly.subplots import make_subplots  # Multi-panel charts
from IPython.display import display, HTML  # Notebook output

print('Libraries loaded successfully.')

In [ ]:
# =============================================================================
# CONFIGURATION
# Central configuration – adjust here, not in the code below

FILE_PATH = 'Beispieldaten_AB_Test.csv'   # Path to the CSV file
ALPHA = 0.05                               # Significance level for t-tests
SCENARIOS = ['Marketing', 'Pricing', 'Prozess']  # Expected scenario names
GROUPS = ['A', 'B']                        # Expected group names

In [ ]:
# =============================================================================
# DATA LOADING & CLEANING

def load_and_validate(filepath: str) -> pd.DataFrame:
    """
    Load CSV, validate required columns, basic data quality checks.
    
    Args:
        filepath: Path to the CSV file
    Returns:
        Cleaned DataFrame
    Raises:
        FileNotFoundError, ValueError for missing columns
    """
    required_columns = ['scenario', 'group', 'units', 'conversions',
                        'revenue', 'costs', 'conversion_rate', 'cost_per_unit']
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        raise FileNotFoundError(f'CSV not found: {filepath}. '
                                'Make sure the file is in the same folder as this notebook.')
    
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    
    print(f'Loaded {len(df):,} rows | {df["scenario"].nunique()} scenarios | '
          f'{df["group"].nunique()} groups')
    print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0].to_string()}')
    return df


df = load_and_validate(FILE_PATH)
display(df.head(10))

In [ ]:
# =============================================================================
# DESCRIPTIVE STATISTICS PER SCENARIO AND GROUP

def describe_by_scenario_group(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate mean KPIs grouped by scenario and group (A vs B).
    
    Returns:
        DataFrame with mean values per scenario/group combination
    """
    metrics = {
        'conversion_rate': 'mean',
        'cost_per_unit': 'mean',
        'revenue': 'mean',
        'costs': 'mean',
        'cycle_time_days': 'mean',
        'errors': 'mean'
    }
    summary = df.groupby(['scenario', 'group']).agg(metrics).round(3)
    return summary


summary = describe_by_scenario_group(df)
print('\n=== DURCHSCHNITTSWERTE JE SZENARIO UND GRUPPE ===')
display(summary)

In [ ]:
# =============================================================================
# STATISTICAL SIGNIFICANCE TEST (t-Test)

def run_ttest(df: pd.DataFrame, scenario: str, metric: str, alpha: float = 0.05) -> dict:
    """
    Run independent samples t-test for a given scenario and metric.
    
    Args:
        df: Input DataFrame
        scenario: Scenario name ('Marketing', 'Pricing', 'Prozess')
        metric: Column name to test
        alpha: Significance level
    Returns:
        dict with test results
    """
    sub = df[df['scenario'] == scenario].dropna(subset=[metric])
    group_a = sub[sub['group'] == 'A'][metric]
    group_b = sub[sub['group'] == 'B'][metric]
    
    if len(group_a) < 2 or len(group_b) < 2:
        return {'scenario': scenario, 'metric': metric, 'significant': 'n/a', 'p_value': None}
    
    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    delta_pct = (group_b.mean() - group_a.mean()) / group_a.mean() * 100 if group_a.mean() != 0 else 0
    
    return {
        'scenario': scenario,
        'metric': metric,
        'mean_A': round(group_a.mean(), 4),
        'mean_B': round(group_b.mean(), 4),
        'delta_%': round(delta_pct, 2),
        'p_value': round(p_value, 4),
        'significant': 'JA ✓' if p_value < alpha else 'Nein',
        'winner': 'B' if group_b.mean() > group_a.mean() else 'A'
    }


# Run t-tests for key metrics per scenario
results = []
test_config = {
    'Marketing': ['conversion_rate', 'cost_per_unit', 'revenue'],
    'Pricing':   ['conversion_rate', 'cost_per_unit', 'revenue'],
    'Prozess':   ['cycle_time_days', 'errors', 'cost_per_unit']
}

for scenario, metrics in test_config.items():
    for metric in metrics:
        results.append(run_ttest(df, scenario, metric, ALPHA))

results_df = pd.DataFrame(results)
print('\n=== STATISTISCHER SIGNIFIKANZTEST (t-Test, Alpha = 5%) ===')
display(results_df)

In [ ]:
# =============================================================================
# VISUALIZATION: Grouped Bar Chart per Scenario

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Marketing: Conversion Rate', 'Pricing: Revenue (€)', 'Prozess: Cycle Time (Tage)'],
    shared_yaxes=False
)

plot_config = [
    ('Marketing', 'conversion_rate', 1),
    ('Pricing',   'revenue',         2),
    ('Prozess',   'cycle_time_days', 3)
]

colors = {'A': '#4C72B0', 'B': '#DD8452'}

for scenario, metric, col in plot_config:
    sub = df[df['scenario'] == scenario].dropna(subset=[metric])
    for grp in ['A', 'B']:
        vals = sub[sub['group'] == grp][metric]
        fig.add_trace(
            go.Bar(
                name=f'Gruppe {grp}',
                x=[f'Gruppe {grp}'],
                y=[vals.mean()],
                error_y=dict(type='data', array=[vals.std()], visible=True),
                marker_color=colors[grp],
                showlegend=(col == 1)
            ),
            row=1, col=col
        )

fig.update_layout(
    title='A/B-Test Ergebnisse: Gruppe A vs. Gruppe B (Mittelwert ± Std.abw.)',
    height=450,
    template='plotly_white',
    barmode='group'
)

fig.show()
fig.write_html('AB_Test_Ergebnisse.html')
print('Chart gespeichert als AB_Test_Ergebnisse.html')

In [ ]:
# =============================================================================
# EXECUTIVE SUMMARY

def generate_executive_summary(results_df: pd.DataFrame) -> str:
    """
    Generate a plain-text executive summary from t-test results.
    
    Returns:
        Formatted summary string
    """
    lines = ['=' * 60]
    lines.append('EXECUTIVE SUMMARY – A/B-TEST-ANALYSE')
    lines.append('=' * 60)
    
    for scenario in results_df['scenario'].unique():
        sub = results_df[results_df['scenario'] == scenario]
        sig_results = sub[sub['significant'] == 'JA ✓']
        lines.append(f'\n{scenario.upper()}:')
        
        if len(sig_results) == 0:
            lines.append('  → Kein statistisch signifikanter Unterschied zwischen A und B.')
        else:
            for _, row in sig_results.iterrows():
                direction = 'besser' if row['winner'] == 'B' else 'schlechter'
                lines.append(f"  → {row['metric']}: Gruppe B {direction} ({row['delta_%']:+.1f}%), "
                             f"p={row['p_value']}")
    
    lines.append('\n' + '=' * 60)
    lines.append('Hinweis: Alle Berechnungen auf Basis synthetischer Dummy-Daten.')
    lines.append('Keine Rückschlüsse auf reale Unternehmensdaten.')
    return '\n'.join(lines)


print(generate_executive_summary(results_df))

---

## Nächste Schritte

1. **Eigene Daten einlesen:** `FILE_PATH` auf eigene CSV setzen (anonymisiert!)
2. **Weitere Metriken:** `test_config` um eigene KPIs ergänzen
3. **Chart exportieren:** `fig.write_image('chart.png')` (benötigt `kaleido`)
4. **Claude fragen:** Code in Claude einfügen und optimieren lassen

---

*Notebook erstellt im Rahmen der Schulung: KI im Controlling | März 2026*  
*Basierend auf Claude-generiertem Code – immer kritisch prüfen!*